# Routing — 02: Corner cases (failure policies, hints, TRANSFORM chain, BQ reporter)

**Persona:** Operations engineer debugging multi-driver writes.


Four independent recipes in one notebook:

1. **`on_failure` semantics** — `fatal` vs `warn` on a fan-out WRITE.
2. **Routing hints** — filter a WRITE fan-out down to one driver using `hints`.
3. **TRANSFORM chain** — enrich READ output by chaining an asset-stats driver after PG.
4. **BigQuery reporter** — `ItemsBigQueryDriver` as an async ingestion mirror
   using the new Phase-3 reporter mode (`flat` / `batch_summary`).

All recipes share the bootstrap cell below; each section is runnable standalone
once the catalog+collection exist.

In [ ]:
import os
import json
import uuid as _uuid
import time as _t

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL    = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")
ADMIN_TOKEN = (
    os.environ.get("DYNASTORE_ADMIN_TOKEN")
    or os.environ.get("DYNASTORE_SYSADMIN_TOKEN")
    or ""
)

headers = {"Authorization": f"Bearer {ADMIN_TOKEN}"} if ADMIN_TOKEN else {}
client  = httpx.Client(base_url=BASE_URL, headers=headers, timeout=60.0)

CATALOG_ID    = f"rtng02-{_uuid.uuid4().hex[:8]}"
COLLECTION_ID = "sensors"
print(f"Using catalog={CATALOG_ID} collection={COLLECTION_ID}")

# Minimal catalog+collection setup (no driver config yet — recipes pin their own)
catalog = {"id": CATALOG_ID, "type": "Catalog", "title": "Routing corner cases",
           "description": "fatal/warn, hints, TRANSFORM, BQ reporter",
           "stac_version": "1.1.0", "conformsTo": [], "links": []}
for _ in range(3):
    r = client.post("/stac/catalogs", content=json.dumps(catalog))
    if r.status_code in (200, 201, 409):
        break
collection = {"id": COLLECTION_ID, "type": "Collection", "title": "Sensors",
              "description": "Demo", "stac_version": "1.1.0",
              "extent": {"spatial": {"bbox": [[-180, -90, 180, 90]]},
                         "temporal": {"interval": [[None, None]]}},
              "license": "proprietary", "keywords": [], "links": []}
r = client.post(f"/stac/catalogs/{CATALOG_ID}/collections", content=json.dumps(collection))
assert r.status_code in (200, 201, 409), r.text[:400]
print("Catalog + collection ready")


## Recipe 1 — `on_failure`: fatal vs warn

Two drivers on the WRITE path, second one is known-misconfigured.
- `fatal` first-driver failure = whole WRITE aborts.
- `warn` second-driver failure = logs + continues; first-driver commit holds.

In [ ]:
# Routing: PG primary (fatal) + DuckDB with an unreachable warehouse (warn).
# The unreachable path makes DuckDB write fail — but the WRITE succeeds overall
# because DuckDB is pinned as on_failure=warn.

routing = {
    "class_key": "CollectionRoutingConfig",
    "operations": {
        "WRITE": [
            {"driver_id": "ItemsPostgresqlDriver", "on_failure": "fatal"},
            {"driver_id": "ItemsDuckdbDriver",     "on_failure": "warn"},
        ],
        "READ":  [{"driver_id": "ItemsPostgresqlDriver"}],
    },
}
duck = {"class_key": "ItemsDuckdbDriverConfig", "warehouse": "/nonexistent/disallowed"}
for (cls_key, body) in [("ItemsPostgresqlDriverConfig", {"class_key": "ItemsPostgresqlDriverConfig"}),
                        ("ItemsDuckdbDriverConfig", duck),
                        ("CollectionRoutingConfig", routing)]:
    r = client.put(
        f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/classes/{cls_key}",
        content=json.dumps(body),
    )
    print(f"PUT {cls_key} → {r.status_code}")


In [ ]:
feat = {"type": "Feature", "id": f"s-{_uuid.uuid4().hex[:6]}", "geometry": {"type": "Point", "coordinates": [0, 0]}, "properties": {"kind": "demo"}}
r = client.post(f"/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items", content=json.dumps({"type": "FeatureCollection", "features": [feat]}))
# Expected: 2xx — PG commits, DuckDB fails-and-warns, WRITE returns success.
print(f"fan-out WRITE (warn) → {r.status_code}")


## Recipe 2 — Routing hints: filter fan-out to a single driver

Two drivers registered; a hint on the POST narrows WRITE to one of them.
Useful for replays / targeted re-indexing without touching the other sinks.

In [ ]:
routing = {
    "class_key": "CollectionRoutingConfig",
    "operations": {
        "WRITE": [
            {"driver_id": "ItemsPostgresqlDriver", "hints": ["features"]},
            {"driver_id": "ItemsElasticsearchObfuscatedDriver", "hints": ["obfuscated"], "on_failure": "warn"},
        ],
        "READ":  [{"driver_id": "ItemsPostgresqlDriver"}],
    },
}
r = client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/classes/CollectionRoutingConfig",
    content=json.dumps(routing),
)
print(f"routing put → {r.status_code}")

# Send a write hinting only "features" → PG only, ES skipped.
headers_hint = {**headers, "X-Dynastore-Hint": "features"}
c2 = httpx.Client(base_url=BASE_URL, headers=headers_hint, timeout=60.0)
r = c2.post(
    f"/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    content=json.dumps({"type": "FeatureCollection", "features": [
        {"type": "Feature", "id": "h-1", "geometry": {"type": "Point", "coordinates": [1, 1]}, "properties": {}}
    ]}),
)
c2.close()
print(f"hinted WRITE (features) → {r.status_code}")


## Recipe 3 — TRANSFORM chain on READ

`metadata.operations.TRANSFORM` is an ordered list of drivers that enrich the
collection descriptor at READ time.  AssetPostgresqlDriver is a natural fit —
it publishes derived summaries (asset counts, last ingestion timestamp) into
the collection envelope that MetadataPostgresqlDriver would otherwise miss.

In [ ]:
routing_with_transform = {
    "class_key": "CollectionRoutingConfig",
    "operations": {
        "WRITE": [{"driver_id": "ItemsPostgresqlDriver", "on_failure": "fatal"}],
        "READ":  [{"driver_id": "ItemsPostgresqlDriver"}],
    },
    "metadata": {
        "operations": {
            "READ":      [{"driver_id": "MetadataPostgresqlDriver"}],
            "TRANSFORM": [{"driver_id": "AssetPostgresqlDriver"}],
        },
    },
}
r = client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/classes/CollectionRoutingConfig",
    content=json.dumps(routing_with_transform),
)
print(f"TRANSFORM-chain routing put → {r.status_code}")

# A GET /collections/{id} should now include asset-derived summaries in the
# envelope if any assets are registered for the collection.
r = client.get(f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}")
print(f"collection GET → {r.status_code}")
print(json.dumps(r.json(), indent=2)[:800])


## Recipe 4 — BigQuery reporter (Phase 3 WRITE capability)

`ItemsBigQueryDriver` now implements WRITE in three modes:
- `off` (default) — WRITE is a no-op even when pinned in a routing config.
- `flat`          — one BQ row per feature (entity_id, payload, ingested_at).
- `batch_summary` — one BQ row per write call (row_count, first/last id).

Partial/complete failures are logged as warnings; the PG primary WRITE is
unaffected.  Use `on_failure=warn` on the BQ entry.

In [ ]:
# 4a. Set the reporter config.
bq_cfg = {
    "class_key": "ItemsBigQueryDriverConfig",
    "target":         {"project_id": "demo-project", "dataset_id": "catalog", "table_name": "ingest_mirror"},
    "report_target":  {"project_id": "demo-project", "dataset_id": "catalog", "table_name": "ingest_mirror"},
    "reporter_mode":  "flat",
    "include_payload": True,
    "exclude_fields": ["secret_key"],
}
r = client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/classes/ItemsBigQueryDriverConfig",
    content=json.dumps(bq_cfg),
)
print(f"BQ reporter config put → {r.status_code}")

# 4b. Pin the routing with BQ as a warn-level WRITE fan-out target.
routing_bq = {
    "class_key": "CollectionRoutingConfig",
    "operations": {
        "WRITE": [
            {"driver_id": "ItemsPostgresqlDriver", "on_failure": "fatal"},
            {"driver_id": "ItemsBigQueryDriver",   "on_failure": "warn"},
        ],
        "READ":  [{"driver_id": "ItemsPostgresqlDriver"}],
    },
}
r = client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/classes/CollectionRoutingConfig",
    content=json.dumps(routing_bq),
)
print(f"routing (BQ reporter) put → {r.status_code}")


In [ ]:
# 4c. Write a feature — PG commits synchronously; BQ reporter fires (best-effort).
# If BQ is not reachable from this environment (no credentials), the warn-level
# entry logs on the server side and does NOT abort the PG commit.

feat = {"type": "Feature", "id": f"bq-{_uuid.uuid4().hex[:6]}",
        "geometry": {"type": "Point", "coordinates": [10, 20]},
        "properties": {"sensor": "rain", "reading": 12.5, "secret_key": "SHOULD-NOT-APPEAR"}}
r = client.post(
    f"/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    content=json.dumps({"type": "FeatureCollection", "features": [feat]}),
)
print(f"WRITE (fan-out) → {r.status_code}")
print("If BigQueryService is unregistered or BQ returns errors, check the server logs")
print("for 'BQ reporter WRITE' entries — the WRITE still returns 2xx thanks to on_failure=warn.")


## Teardown

In [ ]:
r = client.delete(f"/stac/catalogs/{CATALOG_ID}")
print(f"DELETE catalog → {r.status_code}")
client.close()
